# Prime Numbers Lab — Notebook 16: Transition Operator, Entropy, and Mixing

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Notebook purpose:** model residue-class transitions as a finite operator and test entropy, mixing, lift, and transition-conditioned residual structure.

**Core frame:**  
Constraint → structure remains under constraint; drift marks invalid assignments; structure may remain recoverable from partial observation.

Notebook 15 decomposed normalized prime-gap residuals by residue class.

Notebook 16 asks whether the remaining local structure can be represented by a transition operator:

\[
P(r_{n+1}=j \mid r_n=i)
\]

where:

\[
r_n = p_n \bmod 30.
\]

## 0. Setup

This notebook follows the established `prime-numbers-lab` template:

1. define one constraint  
2. generate one dataset  
3. measure what remains under constraint  
4. visualize drift / retention / recoverability  
5. export figures, data, notes, and TeX  
6. package results into a root-level export zip

In [ ]:

# Standard library
from pathlib import Path
import json
import math
import zipfile

# Data / compute
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# Notebook identity
NOTEBOOK_ID = "16_transition_operator_entropy_mixing"
NOTEBOOK_TITLE = "Transition Operator, Entropy, and Mixing"
REPO_NAME = "prime-numbers-lab"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]

# Output directories
OUT = Path(NOTEBOOK_ID)
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
TEX_DIR = OUT / "tex"

for d in [FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT.resolve()}")

## 1. Premise

Notebook 15 showed that normalized prime-gap residuals are not uniformly distributed across residue classes.

Notebook 16 turns that observation into a transition model.

The main object is:

\[
P_{ij}=\Pr(r_{n+1}=j \mid r_n=i)
\]

with residues:

\[
r \in \{1,7,11,13,17,19,23,29\}\pmod{30}.
\]

**Short vocabulary:**

- **Transition operator:** finite Markov-style matrix for residue-to-residue movement.
- **Entropy:** how spread out each anchor residue's next-residue distribution is.
- **Mixing:** how quickly repeated transitions approach a stationary distribution.
- **Lift:** over/under-representation compared with an independent next-residue baseline.
- **Residual-weighted transition:** transition mass weighted by normalized-gap residual deviation.

Core question:

> Are prime residue transitions globally high-entropy while still preserving measurable local arithmetic memory?

## 2. Constraint definition

The transition operator is:

\[
P_{ij}=\frac{\#\{p_n\equiv i,\ p_{n+1}\equiv j \pmod{30}\}}{\#\{p_n\equiv i \pmod{30}\}}
\]

A stationary distribution satisfies:

\[
\pi P = \pi
\]

Row entropy is:

\[
H_i = -\frac{1}{\log 8}\sum_j P_{ij}\log P_{ij}
\]

Lift is:

\[
L_{ij}=\frac{P_{ij}}{\Pr(r_{n+1}=j)}
\]

Transition-conditioned residual is:

\[
\Delta_{i\to j}(z)=f_{i\to j}(z)-e^{-z}
\]

In [ ]:

# Notebook-specific parameters

N_MAX = 2_000_000
RANDOM_SEED = 9423

RESIDUES30 = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
RESIDUE_LABELS = [str(r) for r in RESIDUES30]

Z_MAX = 6.0
BIN_COUNT = 70

WINDOW_COUNT = 14
MIN_WINDOW_TRANSITIONS = 250

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "RESIDUES30": RESIDUES30.tolist(),
    "Z_MAX": Z_MAX,
    "BIN_COUNT": BIN_COUNT,
    "WINDOW_COUNT": WINDOW_COUNT,
    "MIN_WINDOW_TRANSITIONS": MIN_WINDOW_TRANSITIONS,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 3. Data generation

Generate primes up to \(N_{\max}\), remove \(2,3,5\), compute consecutive prime gaps, normalized gaps, and mod-30 residue transitions.

In [ ]:

def generate_primes(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.array([], dtype=int)

    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    for i in range(2, int(math.sqrt(n_max)) + 1):
        if sieve[i]:
            sieve[i*i:n_max+1:i] = False

    return np.nonzero(sieve)[0].astype(int)

primes_all = generate_primes(N_MAX)
primes = primes_all[primes_all > 5]

gaps = np.diff(primes)
anchors = primes[:-1]
next_primes = primes[1:]
normalized_gaps = gaps / np.log(anchors)

anchor_residue = anchors % 30
next_residue = next_primes % 30
gap_residue = gaps % 30

valid = np.isin(anchor_residue, RESIDUES30) & np.isin(next_residue, RESIDUES30)

anchors = anchors[valid]
next_primes = next_primes[valid]
gaps = gaps[valid]
normalized_gaps = normalized_gaps[valid]
anchor_residue = anchor_residue[valid]
next_residue = next_residue[valid]
gap_residue = gap_residue[valid]

summary = {
    "n_max": int(N_MAX),
    "prime_count_excluding_2_3_5": int(len(primes)),
    "transition_count": int(len(anchor_residue)),
    "mean_gap": float(np.mean(gaps)),
    "mean_normalized_gap": float(np.mean(normalized_gaps)),
    "std_normalized_gap": float(np.std(normalized_gaps)),
}

summary

## 4. Transition operator construction

Build:

\[
P(r_{n+1}\mid r_n)
\]

and the raw count matrix.

In [ ]:

residue_to_index = {int(r): i for i, r in enumerate(RESIDUES30)}
n_res = len(RESIDUES30)

transition_counts = np.zeros((n_res, n_res), dtype=float)

for a, b in zip(anchor_residue, next_residue):
    transition_counts[residue_to_index[int(a)], residue_to_index[int(b)]] += 1

row_sums = transition_counts.sum(axis=1, keepdims=True)

transition_operator = np.divide(
    transition_counts,
    row_sums,
    out=np.zeros_like(transition_counts),
    where=row_sums > 0
)

transition_operator_df = pd.DataFrame(
    transition_operator,
    index=RESIDUE_LABELS,
    columns=RESIDUE_LABELS,
)

transition_counts_df = pd.DataFrame(
    transition_counts.astype(int),
    index=RESIDUE_LABELS,
    columns=RESIDUE_LABELS,
)

transition_operator_df.round(4)

## 5. Stationary distribution, entropy, lift, and mixing

Compute:

- stationary distribution
- row entropy
- eigenvalue spectrum
- spectral gap proxy
- mixing distances
- lift against independent next-residue baseline

In [ ]:

def stationary_distribution(P: np.ndarray) -> np.ndarray:
    evals, evecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(evals - 1))
    v = np.real(evecs[:, idx])

    if np.any(v < 0):
        v = np.abs(v)

    if v.sum() == 0:
        v = np.ones(len(v))

    return v / v.sum()

def normalized_entropy(p: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    if len(p) == 0:
        return 0.0
    return float(-np.sum(p * np.log(p)) / np.log(n_res))

def total_variation_distance(p: np.ndarray, q: np.ndarray) -> float:
    return float(0.5 * np.sum(np.abs(np.asarray(p) - np.asarray(q))))

stationary_pi = stationary_distribution(transition_operator)
uniform_pi = np.ones(n_res) / n_res

stationary_df = pd.DataFrame({
    "residue": RESIDUES30,
    "stationary_pi": stationary_pi,
    "uniform": uniform_pi,
    "delta_from_uniform": stationary_pi - uniform_pi,
})

stationary_l1_to_uniform = float(np.sum(np.abs(stationary_pi - uniform_pi)))
stationary_l2_to_uniform = float(np.linalg.norm(stationary_pi - uniform_pi))

row_entropy = np.array([normalized_entropy(transition_operator[i]) for i in range(n_res)])
row_max_probability = transition_operator.max(axis=1)
row_argmax = RESIDUES30[np.argmax(transition_operator, axis=1)]

entropy_df = pd.DataFrame({
    "anchor_residue": RESIDUES30,
    "normalized_entropy": row_entropy,
    "max_transition_probability": row_max_probability,
    "most_likely_next_residue": row_argmax,
})

eigenvalues = np.linalg.eigvals(transition_operator)
eigenvalue_abs = np.sort(np.abs(eigenvalues))[::-1]
lambda_1_abs = float(eigenvalue_abs[0])
lambda_2_abs = float(eigenvalue_abs[1])
spectral_gap_proxy = float(1 - lambda_2_abs)

next_residue_probs = np.array([np.mean(next_residue == r) for r in RESIDUES30])
lift_matrix = np.divide(
    transition_operator,
    next_residue_probs.reshape(1, -1),
    out=np.zeros_like(transition_operator),
    where=next_residue_probs.reshape(1, -1) > 0,
)

lift_df = pd.DataFrame(lift_matrix, index=RESIDUE_LABELS, columns=RESIDUE_LABELS)

lift_records = []
for i, a in enumerate(RESIDUES30):
    for j, b in enumerate(RESIDUES30):
        lift_records.append({
            "transition": f"{a}->{b}",
            "anchor_residue": int(a),
            "next_residue": int(b),
            "probability": float(transition_operator[i, j]),
            "count": int(transition_counts[i, j]),
            "lift": float(lift_matrix[i, j]),
        })

lift_records_df = pd.DataFrame(lift_records).sort_values("lift", ascending=False)

max_k = 20
mixing_rows = []
Pk = np.eye(n_res)

for k in range(1, max_k + 1):
    Pk = Pk @ transition_operator
    tv_by_row = np.array([total_variation_distance(Pk[i], stationary_pi) for i in range(n_res)])
    l2_by_row = np.array([np.linalg.norm(Pk[i] - stationary_pi) for i in range(n_res)])

    mixing_rows.append({
        "k": k,
        "max_tv_to_stationary": float(tv_by_row.max()),
        "mean_tv_to_stationary": float(tv_by_row.mean()),
        "max_l2_to_stationary": float(l2_by_row.max()),
        "mean_l2_to_stationary": float(l2_by_row.mean()),
    })

mixing_df = pd.DataFrame(mixing_rows)

spectral_df = pd.DataFrame({
    "metric": ["lambda_1_abs", "lambda_2_abs", "spectral_gap_proxy"],
    "value": [lambda_1_abs, lambda_2_abs, spectral_gap_proxy],
})

stationary_df.round(6), entropy_df.round(6), spectral_df.round(6)

## 6. Transition-conditioned residual structure

For each transition \(i\to j\), compute normalized-gap residual mass against the Exp(1) baseline.

\[
\Delta_{i\to j}(z)=f_{i\to j}(z)-e^{-z}
\]

In [ ]:

def safe_normalize_pdf(pdf: np.ndarray, bin_width: float) -> np.ndarray:
    total = float(np.sum(pdf) * bin_width)
    if total <= 0:
        return pdf
    return pdf / total

def density_curve(values: np.ndarray, bins: np.ndarray):
    hist, edges = np.histogram(values, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, hist

bins = np.linspace(0, Z_MAX, BIN_COUNT + 1)
centers = 0.5 * (bins[:-1] + bins[1:])
bin_width = float(bins[1] - bins[0])

exp_pdf = np.exp(-centers)
exp_pdf = safe_normalize_pdf(exp_pdf, bin_width)

transition_residual_rows = []
transition_residual_grid_rows = []

for a in RESIDUES30:
    for b in RESIDUES30:
        mask = (anchor_residue == a) & (next_residue == b)
        vals = normalized_gaps[mask]

        if len(vals) < 25:
            continue

        hist, _ = np.histogram(vals, bins=bins, density=True)
        hist = safe_normalize_pdf(hist, bin_width)
        delta = hist - exp_pdf

        l1 = float(np.sum(np.abs(delta)) * bin_width)
        l2 = float(np.sqrt(np.sum(delta**2) * bin_width))
        positive_mass = float(np.sum(np.maximum(delta, 0)) * bin_width)
        negative_mass = float(np.sum(np.minimum(delta, 0)) * bin_width)
        tail_bias = float(np.sum(delta[centers >= 3.0]) * bin_width)

        transition_residual_rows.append({
            "transition": f"{a}->{b}",
            "anchor_residue": int(a),
            "next_residue": int(b),
            "count": int(len(vals)),
            "probability": float(transition_operator[residue_to_index[int(a)], residue_to_index[int(b)]]),
            "mean_z": float(np.mean(vals)),
            "std_z": float(np.std(vals)),
            "l1_residual": l1,
            "l2_residual": l2,
            "positive_residual_mass": positive_mass,
            "negative_residual_mass": negative_mass,
            "tail_bias_z_ge_3": tail_bias,
            "tail_p_z_gt_2": float(np.mean(vals > 2)),
            "tail_p_z_gt_3": float(np.mean(vals > 3)),
        })

        for c, h, e, d in zip(centers, hist, exp_pdf, delta):
            transition_residual_grid_rows.append({
                "transition": f"{a}->{b}",
                "anchor_residue": int(a),
                "next_residue": int(b),
                "z_center": float(c),
                "empirical_pdf": float(h),
                "exp_pdf": float(e),
                "delta": float(d),
            })

transition_residual_df = pd.DataFrame(transition_residual_rows).sort_values("l1_residual", ascending=False)
transition_residual_grid_df = pd.DataFrame(transition_residual_grid_rows)

transition_residual_df.head(20)

## 7. Windowed transition drift

Compute transition operators in log-spaced windows and measure:

\[
\|P_x-P\|_1
\]

along with windowed entropy.

In [ ]:

raw_edges = np.unique(np.logspace(np.log10(int(anchors.min())), np.log10(int(anchors.max())), WINDOW_COUNT + 1).astype(int))
raw_edges[0] = int(anchors.min())
raw_edges[-1] = int(anchors.max())

window_rows = []
window_operator_rows = []

for idx, (left, right) in enumerate(zip(raw_edges[:-1], raw_edges[1:]), start=1):
    mask = (anchors >= left) & (anchors < right)

    if int(mask.sum()) < MIN_WINDOW_TRANSITIONS:
        continue

    counts_w = np.zeros((n_res, n_res), dtype=float)

    for a, b in zip(anchor_residue[mask], next_residue[mask]):
        counts_w[residue_to_index[int(a)], residue_to_index[int(b)]] += 1

    row_sums_w = counts_w.sum(axis=1, keepdims=True)
    P_w = np.divide(
        counts_w,
        row_sums_w,
        out=np.zeros_like(counts_w),
        where=row_sums_w > 0,
    )

    entropy_w = np.array([normalized_entropy(P_w[i]) for i in range(n_res)])
    operator_l1_drift = float(np.mean(np.abs(P_w - transition_operator)))

    window_rows.append({
        "window_index": idx,
        "left": int(left),
        "right": int(right),
        "midpoint": float(math.sqrt(left * right)),
        "transition_count": int(mask.sum()),
        "operator_l1_drift": operator_l1_drift,
        "mean_transition_entropy": float(np.mean(entropy_w)),
        "min_transition_entropy": float(np.min(entropy_w)),
        "max_transition_entropy": float(np.max(entropy_w)),
    })

    for i, a in enumerate(RESIDUES30):
        for j, b in enumerate(RESIDUES30):
            window_operator_rows.append({
                "window_index": idx,
                "midpoint": float(math.sqrt(left * right)),
                "anchor_residue": int(a),
                "next_residue": int(b),
                "transition_probability": float(P_w[i, j]),
                "global_transition_probability": float(transition_operator[i, j]),
                "delta_from_global": float(P_w[i, j] - transition_operator[i, j]),
            })

window_transition_df = pd.DataFrame(window_rows)
window_operator_grid_df = pd.DataFrame(window_operator_rows)

window_transition_df

## 8. Residual-weighted transition operator

Weight transition probability by transition-conditioned residual mass:

\[
W_{ij}=P_{ij}\|\Delta_{i\to j}\|_1
\]

Then normalize to show each transition's share of residual-weighted mass.

In [ ]:

residual_matrix = np.zeros((n_res, n_res), dtype=float)

metric_lookup = {
    (int(row.anchor_residue), int(row.next_residue)): float(row.l1_residual)
    for _, row in transition_residual_df.iterrows()
}

for i, a in enumerate(RESIDUES30):
    for j, b in enumerate(RESIDUES30):
        residual_matrix[i, j] = metric_lookup.get((int(a), int(b)), 0.0)

residual_weighted_operator = transition_operator * residual_matrix
total_residual_weight = residual_weighted_operator.sum()

if total_residual_weight > 0:
    residual_weighted_share = residual_weighted_operator / total_residual_weight
else:
    residual_weighted_share = residual_weighted_operator

residual_weighted_df = pd.DataFrame(
    residual_weighted_share,
    index=RESIDUE_LABELS,
    columns=RESIDUE_LABELS,
)

residual_weighted_df.round(5)

## 9. Visualization

Notebook 16 produces transition-operator figures:

1. transition operator heatmap  
2. stationary distribution  
3. entropy by anchor residue  
4. mixing distance vs steps  
5. eigenvalue magnitudes  
6. transition lift heatmap  
7. top transition residual masses  
8. residual-weighted transition operator  
9. transition-conditioned residual curves  
10. windowed operator drift  
11. windowed transition entropy  
12. windowed transition delta heatmap

### Figure 1 — transition operator heatmap

In [ ]:

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(transition_operator, aspect="auto")
ax.set_title("Prime residue transition operator mod30")
ax.set_xlabel("next residue p_{n+1} mod 30")
ax.set_ylabel("anchor residue p_n mod 30")
ax.set_xticks(np.arange(n_res))
ax.set_yticks(np.arange(n_res))
ax.set_xticklabels(RESIDUE_LABELS)
ax.set_yticklabels(RESIDUE_LABELS)
fig.colorbar(im, ax=ax, label="transition probability")

for i in range(n_res):
    for j in range(n_res):
        ax.text(j, i, f"{transition_operator[i,j]:.2f}", ha="center", va="center", fontsize=8)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_transition_operator_heatmap.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

### Figure 2 — stationary distribution

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(RESIDUE_LABELS, stationary_pi, label="stationary distribution")
ax.axhline(1 / n_res, linestyle="--", label="uniform 1/8")
ax.set_title("Stationary distribution")
ax.set_xlabel("residue mod30")
ax.set_ylabel("probability")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_stationary_distribution.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

### Figure 3 — transition entropy by anchor residue

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(RESIDUE_LABELS, row_entropy, marker="o")
ax.axhline(1.0, linestyle="--", label="maximum entropy")
ax.set_title("Transition entropy by anchor residue")
ax.set_xlabel("anchor residue mod30")
ax.set_ylabel("normalized entropy")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_transition_entropy_by_anchor.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

### Figure 4 — mixing distance vs steps

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mixing_df["k"], mixing_df["max_tv_to_stationary"], marker="o", label="max TV distance")
ax.plot(mixing_df["k"], mixing_df["mean_tv_to_stationary"], marker="o", label="mean TV distance")
ax.set_title("Mixing toward stationary distribution")
ax.set_xlabel("transition steps k")
ax.set_ylabel("total variation distance")
ax.legend()
ax.grid(True, alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_mixing_distance_vs_steps.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()

fig4_path

### Figure 5 — eigenvalue magnitudes

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([str(i) for i in range(len(eigenvalue_abs))], eigenvalue_abs)
ax.set_title("Transition operator eigenvalue magnitudes")
ax.set_xlabel("eigenvalue rank")
ax.set_ylabel("|lambda|")
ax.grid(True, axis="y", alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_transition_eigenvalue_magnitudes.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()

fig5_path

### Figure 6 — transition lift heatmap

In [ ]:

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(lift_matrix, aspect="auto")
ax.set_title("Transition lift over independent next-residue baseline")
ax.set_xlabel("next residue mod30")
ax.set_ylabel("anchor residue mod30")
ax.set_xticks(np.arange(n_res))
ax.set_yticks(np.arange(n_res))
ax.set_xticklabels(RESIDUE_LABELS)
ax.set_yticklabels(RESIDUE_LABELS)
fig.colorbar(im, ax=ax, label="lift ratio")

for i in range(n_res):
    for j in range(n_res):
        ax.text(j, i, f"{lift_matrix[i,j]:.2f}", ha="center", va="center", fontsize=8)

fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_transition_lift_heatmap.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()

fig6_path

### Figure 7 — top transition residual masses

In [ ]:

top = transition_residual_df.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top["transition"], top["l1_residual"])
ax.set_title("Top transition residual masses")
ax.set_xlabel("L1 residual vs Exp(1)")
ax.set_ylabel("transition")
ax.grid(True, axis="x", alpha=0.3)

fig7_path = FIG_DIR / f"{NOTEBOOK_NUM}_top_transition_residual_masses.png"
fig.savefig(fig7_path, dpi=180, bbox_inches="tight")
plt.show()

fig7_path

### Figure 8 — residual-weighted transition operator

In [ ]:

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(residual_weighted_share, aspect="auto")
ax.set_title("Residual-weighted transition contribution")
ax.set_xlabel("next residue mod30")
ax.set_ylabel("anchor residue mod30")
ax.set_xticks(np.arange(n_res))
ax.set_yticks(np.arange(n_res))
ax.set_xticklabels(RESIDUE_LABELS)
ax.set_yticklabels(RESIDUE_LABELS)
fig.colorbar(im, ax=ax, label="share of residual-weighted mass")

for i in range(n_res):
    for j in range(n_res):
        ax.text(j, i, f"{residual_weighted_share[i,j]:.3f}", ha="center", va="center", fontsize=7)

fig8_path = FIG_DIR / f"{NOTEBOOK_NUM}_residual_weighted_transition_operator.png"
fig.savefig(fig8_path, dpi=180, bbox_inches="tight")
plt.show()

fig8_path

### Figure 9 — strongest transition-conditioned residual curves

In [ ]:

top_transitions = transition_residual_df.head(6)["transition"].tolist()

fig, ax = plt.subplots(figsize=(10, 6))

for transition in top_transitions:
    a, b = [int(x) for x in transition.split("->")]
    sub = transition_residual_grid_df[transition_residual_grid_df["transition"] == transition]
    ax.plot(sub["z_center"], sub["delta"], label=transition)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Strongest transition-conditioned residual curves")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("Delta(z)")
ax.legend()
ax.grid(True, alpha=0.3)

fig9_path = FIG_DIR / f"{NOTEBOOK_NUM}_transition_conditioned_residual_curves.png"
fig.savefig(fig9_path, dpi=180, bbox_inches="tight")
plt.show()

fig9_path

### Figure 10 — windowed operator drift

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(window_transition_df["midpoint"], window_transition_df["operator_l1_drift"], marker="o")
ax.set_xscale("log")
ax.set_title("Windowed transition-operator drift")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("mean |P_window - P_global|")
ax.grid(True, alpha=0.3)

fig10_path = FIG_DIR / f"{NOTEBOOK_NUM}_windowed_transition_operator_drift.png"
fig.savefig(fig10_path, dpi=180, bbox_inches="tight")
plt.show()

fig10_path

### Figure 11 — windowed transition entropy

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(window_transition_df["midpoint"], window_transition_df["mean_transition_entropy"], marker="o", label="mean entropy")
ax.plot(window_transition_df["midpoint"], window_transition_df["min_transition_entropy"], marker="o", label="min entropy")
ax.axhline(1.0, linestyle="--", label="maximum")
ax.set_xscale("log")
ax.set_title("Windowed transition entropy")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("normalized entropy")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

fig11_path = FIG_DIR / f"{NOTEBOOK_NUM}_windowed_transition_entropy.png"
fig.savefig(fig11_path, dpi=180, bbox_inches="tight")
plt.show()

fig11_path

### Figure 12 — windowed transition delta heatmap

In [ ]:

transition_labels = [f"{a}->{b}" for a in RESIDUES30 for b in RESIDUES30]

grid = window_operator_grid_df.copy()
grid["transition"] = grid["anchor_residue"].astype(str) + "->" + grid["next_residue"].astype(str)

pivot_delta = grid.pivot_table(index="window_index", columns="transition", values="delta_from_global", aggfunc="mean")
pivot_delta = pivot_delta.reindex(columns=transition_labels)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot_delta.values, aspect="auto", interpolation="nearest", origin="lower")
ax.set_title("Windowed transition delta from global operator")
ax.set_xlabel("transition")
ax.set_ylabel("window index")
ax.set_xticks(np.arange(len(transition_labels))[::4])
ax.set_xticklabels(transition_labels[::4], rotation=90, fontsize=7)
fig.colorbar(im, ax=ax, label="P_window - P_global")

fig12_path = FIG_DIR / f"{NOTEBOOK_NUM}_windowed_transition_delta_heatmap.png"
fig.savefig(fig12_path, dpi=180, bbox_inches="tight")
plt.show()

fig12_path

## 10. Interpretation

Use a short, consistent structure:

1. **What remains under constraint?**  
2. **What drifts?**  
3. **What appears recoverable?**  
4. **What should not be overclaimed?**

In [ ]:

top_lift = lift_records_df.iloc[0]
top_residual = transition_residual_df.iloc[0]
min_entropy_row = entropy_df.sort_values("normalized_entropy").iloc[0]
max_entropy_row = entropy_df.sort_values("normalized_entropy", ascending=False).iloc[0]

measurement = {
    "stationary_l1_to_uniform": stationary_l1_to_uniform,
    "stationary_l2_to_uniform": stationary_l2_to_uniform,
    "mean_transition_entropy": float(row_entropy.mean()),
    "min_transition_entropy": float(row_entropy.min()),
    "max_transition_entropy": float(row_entropy.max()),
    "lambda_2_abs": lambda_2_abs,
    "spectral_gap_proxy": spectral_gap_proxy,
    "top_lift_transition": str(top_lift["transition"]),
    "top_lift_value": float(top_lift["lift"]),
    "top_residual_transition": str(top_residual["transition"]),
    "top_residual_l1": float(top_residual["l1_residual"]),
    "windowed_operator_drift_min": float(window_transition_df["operator_l1_drift"].min()),
    "windowed_operator_drift_max": float(window_transition_df["operator_l1_drift"].max()),
}

cgcs_score = 1.0 / (
    1.0
    + measurement["stationary_l1_to_uniform"]
    + (1.0 - measurement["mean_transition_entropy"])
    + measurement["lambda_2_abs"]
)

cgcs = {
    "score": float(cgcs_score),
    "definition": "1/(1 + stationary L1 to uniform + entropy deficit + lambda2)",
    "interpretation": "Higher score indicates closer-to-uniform stationary behavior, higher entropy, and faster mixing.",
}

interpretation = f'''
# {NOTEBOOK_TITLE}

## Constraint result

This notebook models residue-class transitions as a finite transition operator.

The transition operator is:

$$
P_{{ij}} = \\Pr(r_{{n+1}}=j \\mid r_n=i).
$$

## Remains under constraint

Prime residue transitions form a high-entropy operator, meaning the next residue is broadly distributed for each anchor residue.

Measured values:

- mean transition entropy = {measurement["mean_transition_entropy"]:.6f}
- stationary L1 distance from uniform = {measurement["stationary_l1_to_uniform"]:.6f}
- spectral gap proxy = {measurement["spectral_gap_proxy"]:.6f}

## Drift

Drift appears as:

- transition lift above independent next-residue baseline
- residual-weighted transition mass
- transition-conditioned residual curves
- windowed transition-operator drift

Strongest lift transition:

- {measurement["top_lift_transition"]}, lift = {measurement["top_lift_value"]:.6f}

Strongest transition residual:

- {measurement["top_residual_transition"]}, L1 = {measurement["top_residual_l1"]:.6f}

## Recoverability

The local structure is recoverable through:

- transition matrix
- entropy profile
- mixing profile
- lift matrix
- residual-weighted transition operator
- transition-conditioned residual curves

## CGCS score

The transition-mixing score is:

$$
CGCS_{{transition}} =
\\frac{{1}}{{1 + d(\\pi,U) + (1-\\overline{{H}}) + \\lambda_2}}.
$$

Measured score:

$$
CGCS_{{transition}} = {cgcs_score:.6f}.
$$

## Caution

This notebook does not claim prime residues are generated by a first-order Markov chain.

It uses a finite transition operator as a measurement tool for local arithmetic memory.
'''.strip()

print(interpretation)

## 11. Export data, notes, figures index, math, and TeX

This block writes reusable artifacts:

- CSV summaries
- JSON metadata
- Markdown interpretation with embedded figure links
- Markdown design notes
- TeX results snippet
- standalone TeX math notes

In [ ]:

summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
transition_operator_path = DATA_DIR / f"{NOTEBOOK_NUM}_transition_operator.csv"
transition_counts_path = DATA_DIR / f"{NOTEBOOK_NUM}_transition_counts.csv"
stationary_path = DATA_DIR / f"{NOTEBOOK_NUM}_stationary_distribution.csv"
entropy_path = DATA_DIR / f"{NOTEBOOK_NUM}_entropy_by_anchor.csv"
spectral_path = DATA_DIR / f"{NOTEBOOK_NUM}_spectral_metrics.csv"
mixing_path = DATA_DIR / f"{NOTEBOOK_NUM}_mixing_metrics.csv"
lift_path = DATA_DIR / f"{NOTEBOOK_NUM}_transition_lift_records.csv"
transition_residual_path = DATA_DIR / f"{NOTEBOOK_NUM}_transition_residual_metrics.csv"
transition_residual_grid_path = DATA_DIR / f"{NOTEBOOK_NUM}_transition_residual_grid.csv"
window_transition_path = DATA_DIR / f"{NOTEBOOK_NUM}_windowed_transition_drift.csv"
window_operator_grid_path = DATA_DIR / f"{NOTEBOOK_NUM}_windowed_transition_operator_grid.csv"
residual_weighted_path = DATA_DIR / f"{NOTEBOOK_NUM}_residual_weighted_transition_operator.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_notes_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
transition_operator_df.to_csv(transition_operator_path)
transition_counts_df.to_csv(transition_counts_path)
stationary_df.to_csv(stationary_path, index=False)
entropy_df.to_csv(entropy_path, index=False)
spectral_df.to_csv(spectral_path, index=False)
mixing_df.to_csv(mixing_path, index=False)
lift_records_df.to_csv(lift_path, index=False)
transition_residual_df.to_csv(transition_residual_path, index=False)
transition_residual_grid_df.to_csv(transition_residual_grid_path, index=False)
window_transition_df.to_csv(window_transition_path, index=False)
window_operator_grid_df.to_csv(window_operator_grid_path, index=False)
residual_weighted_df.to_csv(residual_weighted_path)

figure_paths = [
    fig1_path,
    fig2_path,
    fig3_path,
    fig4_path,
    fig5_path,
    fig6_path,
    fig7_path,
    fig8_path,
    fig9_path,
    fig10_path,
    fig11_path,
    fig12_path,
]

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "transition_operator": str(transition_operator_path),
        "transition_counts": str(transition_counts_path),
        "stationary_distribution": str(stationary_path),
        "entropy_by_anchor": str(entropy_path),
        "spectral_metrics": str(spectral_path),
        "mixing_metrics": str(mixing_path),
        "transition_lift_records": str(lift_path),
        "transition_residual_metrics": str(transition_residual_path),
        "transition_residual_grid": str(transition_residual_grid_path),
        "windowed_transition_drift": str(window_transition_path),
        "windowed_transition_operator_grid": str(window_operator_grid_path),
        "residual_weighted_transition_operator": str(residual_weighted_path),
    },
    "docs": {
        "interpretation": str(interpretation_md_path),
        "design_notes": str(design_notes_md_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

figures_md = "\n\n## Figures\n\n"
for i, fig in enumerate(figure_paths, start=1):
    figures_md += f"### Figure {i} — {fig.stem.replace('_', ' ').title()}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

interpretation_md_path.write_text(interpretation + figures_md, encoding="utf-8")

design_notes = f'''
# Design Notes — {NOTEBOOK_TITLE}

## Notebook role

Notebook 16 follows Notebook 15 by turning residue-class residual structure into a transition operator.

## Constraint

The main object is:

$$
P_{{ij}}=\\Pr(r_{{n+1}}=j\\mid r_n=i).
$$

## Measurement

Metrics include stationary distribution, entropy, spectral gap proxy, mixing distance, transition lift, transition-conditioned residuals, residual-weighted transition mass, and windowed transition drift.

## CGCS score

$$
CGCS_{{transition}} =
\\frac{{1}}{{1+d(\\pi,U)+(1-\\overline{{H}})+\\lambda_2}}.
$$

## Handoff

Notebook 17 should test higher-order transitions or compare first-order transition memory against shuffled baselines.
'''.strip()

design_notes_md_path.write_text(design_notes + "\n", encoding="utf-8")

summary_tex = rf'''
\section*{{{NOTEBOOK_TITLE}}}

This notebook models prime residue transitions as a finite transition operator.

\[
P_{{ij}}=\Pr(r_{{n+1}}=j\mid r_n=i)
\]

\begin{{itemize}}
  \item Transition count: {summary["transition_count"]}
  \item Mean transition entropy: {measurement["mean_transition_entropy"]:.6f}
  \item Stationary L1 distance from uniform: {measurement["stationary_l1_to_uniform"]:.6f}
  \item $\lambda_2$: {measurement["lambda_2_abs"]:.6f}
  \item Spectral gap proxy: {measurement["spectral_gap_proxy"]:.6f}
  \item Strongest lift transition: {measurement["top_lift_transition"]}
  \item Strongest transition residual: {measurement["top_residual_transition"]}
  \item CGCS transition score: {cgcs_score:.6f}
\end{{itemize}}

The transition operator is treated as a finite-scale measurement of local arithmetic memory.
'''.strip()

summary_tex_path.write_text(summary_tex + "\n", encoding="utf-8")

math_tex = rf'''
\documentclass{{article}}
\usepackage{{amsmath}}
\usepackage{{amssymb}}
\usepackage[margin=1in]{{geometry}}

\begin{{document}}

\section*{{Math Notes: {NOTEBOOK_TITLE}}}

\subsection*{{Residue transition operator}}

\[
P_{{ij}}=\Pr(r_{{n+1}}=j\mid r_n=i)
\]

\subsection*{{Stationary distribution}}

\[
\pi P = \pi
\]

\subsection*{{Entropy}}

\[
H_i = -\frac{{1}}{{\log 8}}\sum_j P_{{ij}}\log P_{{ij}}
\]

\subsection*{{Lift}}

\[
L_{{ij}} = \frac{{P_{{ij}}}}{{\Pr(r_{{n+1}}=j)}}
\]

\subsection*{{Transition-conditioned residual}}

\[
\Delta_{{i\to j}}(z)=f_{{i\to j}}(z)-e^{{-z}}
\]

\subsection*{{Spectral gap proxy}}

\[
gap = 1-\lambda_2
\]

\subsection*{{CGCS transition score}}

\[
CGCS_{{transition}} =
\frac{{1}}{{1+d(\pi,U)+(1-\overline{{H}})+\lambda_2}}
\]

\end{{document}}
'''.strip()

math_tex_path.write_text(math_tex + "\n", encoding="utf-8")

summary_path, transition_operator_path, transition_counts_path, stationary_path, entropy_path, spectral_path, mixing_path, lift_path, transition_residual_path, window_transition_path, metadata_path, interpretation_md_path, design_notes_md_path, summary_tex_path, math_tex_path

## 12. Optional results bundle

This creates a root-level export zip containing figures, data, docs, and TeX outputs.

In [ ]:

EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 13. Next notebook handoff

Next:

> Notebook 17 should test higher-order transition memory or compare first-order residue transitions against shuffled baselines.

In [ ]:

next_step = "Notebook 17: higher-order transition memory and shuffled baselines."
print(next_step)